In [1]:
import shutil
import os

# Define source and destination paths
repo_src = '/kaggle/input/pipnet-github-repository'
repo_dst = '/kaggle/working/PIPNet'

# Remove the existing directory to prevent errors during the copy process
if os.path.exists(repo_dst):
    shutil.rmtree(repo_dst)

# Copy the repository to the working directory
# Note: Ensure repo_src points directly to the folder containing 'setup.py' or 'main.py'
shutil.copytree(repo_src, repo_dst)

print(f"Repository ready at {repo_dst}")

Repository ready at /kaggle/working/PIPNet


In [2]:
%%writefile /kaggle/working/PIPNet/util/data.py
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import os

"""
[MODIFICATION: CLASS MOVED TO TOP]
Original Location: End of the file.
Reason: Moved 'TwoAugSupervisedDataset' to the top to ensure it is defined before being instantiated 
        in the 'get_covid' function. This prevents NameError in the script execution flow.
"""
class TwoAugSupervisedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform1, transform2):
        self.dataset = dataset
        self.transform1 = transform1
        self.transform2 = transform2
        # Handle cases where dataset is a Subset or original ImageFolder
        if hasattr(dataset, 'classes'):
            self.classes = dataset.classes
        else:
            self.classes = dataset.dataset.classes

    def __getitem__(self, index):
        image, target = self.dataset[index]
        return self.transform1(image), self.transform2(image), target

    def __len__(self):
        return len(self.dataset)

"""
[MODIFICATION: NEW FUNCTION 'get_covid']
Reason: 
    Added specifically for Phase 5 to handle the COVID-19 Radiography Database.
    This function replaces the generic 'create_datasets' logic for this specific use case.

Key Changes:
    1. Channel Conversion: Converts 1-channel Grayscale X-Rays to 3-channel RGB to match 
       the input requirements of the pre-trained ResNet backbone.
    2. Data Split Protocol: Implements the exact validation protocol from Phase 3 
       (e.g., 75% Train/Val split logic) using 'train_test_split'.
    3. Augmentation: Defines separate transforms for training (with augmentation) and testing.
"""
def get_covid(root_dir: str, img_size: int, seed: int):
    # Standard ImageNet mean and std (as established in Phase 3 protocol)
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    normalize = transforms.Normalize(mean=mean, std=std)
    
    # Transform for Test/Validation (No augmentation, just resize & normalize)
    transform_no_augment = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.Grayscale(3), # Convert to 3 channels
        transforms.ToTensor(), 
        normalize
    ])
    
    # Transform for Training (With visual-preserving augmentations)
    transform_aug = transforms.Compose([
        transforms.Resize((img_size + 32, img_size + 32)),
        transforms.RandomHorizontalFlip(), 
        transforms.RandomRotation(15),
        transforms.RandomResizedCrop(img_size, scale=(0.9, 1.0)), 
        transforms.Grayscale(3), # Convert to 3 channels
        transforms.ToTensor(), 
        normalize
    ])

    # Load base dataset
    base_dataset = torchvision.datasets.ImageFolder(root_dir)
    indices = np.arange(len(base_dataset))
    targets = np.array(base_dataset.targets)

    # Stratified Split (Phase 3 Protocol)
    # Splitting Train+Val (85%) and Test (15%)
    train_val_idx, test_idx = train_test_split(indices, test_size=0.15, stratify=targets, random_state=seed)
    # Further splitting Train (approx 65% total) and Val (approx 15% total)
    # 0.10 / 0.85 is roughly 11.7% of the remaining data, adjusting to match protocol
    train_idx, val_idx = train_test_split(train_val_idx, test_size=(0.10/0.85), stratify=targets[train_val_idx], random_state=seed)

    # Creating Subsets
    trainset = TwoAugSupervisedDataset(Subset(base_dataset, train_idx), transform_aug, transform_aug)
    testset = Subset(torchvision.datasets.ImageFolder(root_dir, transform=transform_no_augment), test_idx)
    valset = Subset(torchvision.datasets.ImageFolder(root_dir, transform=transform_no_augment), val_idx)
    
    # Project set: Training data without augmentation (for prototype visualization)
    projectset = Subset(torchvision.datasets.ImageFolder(root_dir, transform=transform_no_augment), train_idx)

    return trainset, testset, valset, projectset, base_dataset.classes

"""
[MODIFICATION: CUSTOMIZED 'get_dataloaders']
Original: Used 'args.dataset' to switch between CUB, PETS, etc.
Modified: 
    1. Hardcoded logic for 'COVID19' to point to the specific Kaggle input directory.
    2. Integrated 'WeightedRandomSampler' to address class imbalance (Normal vs COVID/Pneumonia).
    3. Modified Return Values: Returns 'None' for 'trainloader_normal' etc., as they are not 
       used in the optimized training loop of Phase 5.
"""
def get_dataloaders(args, device):
    # Handle Kaggle directory structure (sometimes zip extracts to a subfolder)
    covid_path = '/kaggle/input/pipnet-project-data/COVID19/COVID19'
    if not os.path.exists(covid_path): 
        covid_path = '/kaggle/input/pipnet-project-data/COVID19'
    
    # Call the custom data preparation function
    trainset, testset, valset, projectset, classes = get_covid(covid_path, args.image_size, args.seed)
    
    # [MODIFICATION] Imbalanced Data Handling
    sampler = None
    if args.weighted_loss:
        # Extract targets from the Subset to calculate inverse frequency weights
        train_targets = np.array([trainset.dataset.dataset.targets[i] for i in trainset.dataset.indices])
        counts = np.bincount(train_targets)
        weights = 1. / counts
        sample_weights = weights[train_targets]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    dl_args = {'batch_size': args.batch_size, 'num_workers': args.num_workers, 'pin_memory': True}
    
    # Create DataLoaders
    # Note: Shuffle must be False if Sampler is provided
    trainloader = DataLoader(trainset, shuffle=(sampler is None), sampler=sampler, drop_last=True, **dl_args)
    
    # Pretraining loader (usually same as train loader in this implementation)
    trainloader_pre = DataLoader(trainset, batch_size=args.batch_size_pretrain, shuffle=(sampler is None), sampler=sampler, drop_last=True, num_workers=args.num_workers, pin_memory=True)
    
    testloader = DataLoader(testset, shuffle=False, **dl_args)
    valloader = DataLoader(valset, shuffle=False, **dl_args)
    projectloader = DataLoader(projectset, batch_size=1, shuffle=False)

    # Returning None for unused loaders to maintain function signature compatibility with main.py unpacking
    return trainloader, trainloader_pre, None, None, projectloader, testloader, valloader, classes

Overwriting /kaggle/working/PIPNet/util/data.py


In [3]:
%%writefile /kaggle/working/PIPNet/util/args.py
import os
import argparse
import pickle
import numpy as np
import random
import torch

def get_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser('Train a PIP-Net')
    
    # --- Standard Arguments ---
    parser.add_argument('--dataset', type=str, default='CUB-200-2011')
    parser.add_argument('--net', type=str, default='resnet18')
    parser.add_argument('--batch_size', type=int, default=64)
    parser.add_argument('--batch_size_pretrain', type=int, default=64)
    parser.add_argument('--epochs', type=int, default=60)
    parser.add_argument('--epochs_pretrain', type=int, default=10)
    parser.add_argument('--optimizer', type=str, default='Adam')
    parser.add_argument('--lr', type=float, default=0.05)
    parser.add_argument('--lr_net', type=float, default=0.0005)
    parser.add_argument('--lr_block', type=float, default=0.0005)
    parser.add_argument('--weight_decay', type=float, default=0.0)
    parser.add_argument('--disable_cuda', action='store_true')
    parser.add_argument('--log_dir', type=str, default='./runs/run_pipnet')
    parser.add_argument('--num_features', type=int, default=0)
    parser.add_argument('--image_size', type=int, default=224)
    parser.add_argument('--state_dict_dir_net', type=str, default='')
    parser.add_argument('--freeze_epochs', type=int, default=10)
    parser.add_argument('--dir_for_saving_images', type=str, default='visualization_results')
    parser.add_argument('--disable_pretrained', action='store_true')
    parser.add_argument('--weighted_loss', action='store_true')
    parser.add_argument('--seed', type=int, default=1)
    parser.add_argument('--gpu_ids', type=str, default='')
    parser.add_argument('--num_workers', type=int, default=4)
    parser.add_argument('--bias', action='store_true')
    parser.add_argument('--extra_test_image_folder', type=str, default='')
    parser.add_argument('--validation_size', type=float, default=0.0)
    
    """
    [MODIFICATION: VISUALIZATION SWITCH]
    Added '--visualize' flag.
    Reason: Visualization is computationally expensive. This flag allows skipping it during
            hyperparameter tuning to save time.
    """
    parser.add_argument('--visualize', action='store_true', help='Enable visualization step at the end')

    """
    [MODIFICATION: SPARSITY TUNING KNOBS]
    Added '--decay_rate' and '--decay_start_epoch'.
    Reason: To enable 'Scheduled Smart Pruning'.
            - decay_rate: Controls the magnitude of weight decay (pruning strength).
            - decay_start_epoch: Controls when pruning starts (warm-up period).
            This allows fine-tuning the sparsity mechanism without code modifications.
    """
    parser.add_argument('--decay_rate', type=float, default=0.0008, help='Rate of weight decay for prototypes')
    parser.add_argument('--decay_start_epoch', type=int, default=10, help='Epoch to start decaying weights')

    """
    [MODIFICATION: LOSS WEIGHT HYPERPARAMETERS]
    Added '--w_align', '--w_tanh', '--w_class'.
    Reason: To make the loss function weights tunable from the command line.
            This is essential for sensitivity analysis in Phase 5, allowing you to see
            how different loss components affect the model's learning.
            Default values match the original paper's settings (5.0, 2.0, 2.0).
    """
    parser.add_argument('--w_align', type=float, default=5.0, help='Weight for Alignment Loss (Lambda 1)')
    parser.add_argument('--w_tanh', type=float, default=2.0, help='Weight for Tanh/Uniformity Loss (Lambda 2)')
    parser.add_argument('--w_class', type=float, default=2.0, help='Weight for Classification Loss (Lambda 3)')

    args = parser.parse_args()
    if len(args.log_dir.split('/'))>2:
        if not os.path.exists(args.log_dir):
            os.makedirs(args.log_dir)
    return args

def save_args(args: argparse.Namespace, directory_path: str) -> None:
    if not os.path.isdir(directory_path):
        os.mkdir(directory_path)
    with open(directory_path + '/args.txt', 'w') as f:
        for arg in vars(args):
            val = getattr(args, arg)
            if isinstance(val, str):
                val = f"'{val}'"
            f.write('{}: {}\n'.format(arg, val))
    with open(directory_path + '/args.pickle', 'wb') as f:
        pickle.dump(args, f)

def get_optimizer_nn(net, args: argparse.Namespace) -> torch.optim.Optimizer:
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)

    params_to_freeze = []
    params_to_train = []
    params_backbone = []
    
    # Simplified logic for optimizer parameter grouping
    if 'resnet' in args.net:
        for name, param in net.module._net.named_parameters():
            if 'layer4' in name:
                params_to_train.append(param)
            else:
                params_to_freeze.append(param)
    else:
        for name, param in net.module._net.named_parameters():
            params_backbone.append(param)

    classification_weight = []
    classification_bias = []
    for name, param in net.module._classification.named_parameters():
        if 'weight' in name:
            classification_weight.append(param)
        elif 'bias' in name and args.bias:
            classification_bias.append(param)

    paramlist_net = [
        {"params": params_backbone, "lr": args.lr_net, "weight_decay_rate": args.weight_decay},
        {"params": params_to_freeze, "lr": args.lr_block, "weight_decay_rate": args.weight_decay},
        {"params": params_to_train, "lr": args.lr_block, "weight_decay_rate": args.weight_decay},
        {"params": net.module._add_on.parameters(), "lr": args.lr_block*10., "weight_decay_rate": args.weight_decay}
    ]
    paramlist_classifier = [
        {"params": classification_weight, "lr": args.lr, "weight_decay_rate": args.weight_decay},
        {"params": classification_bias, "lr": args.lr, "weight_decay_rate": 0},
    ]

    optimizer_net = torch.optim.AdamW(paramlist_net, lr=args.lr_net, weight_decay=args.weight_decay)
    optimizer_classifier = torch.optim.AdamW(paramlist_classifier, lr=args.lr, weight_decay=args.weight_decay)
    return optimizer_net, optimizer_classifier, params_to_freeze, params_to_train, params_backbone

Overwriting /kaggle/working/PIPNet/util/args.py


In [4]:
%%writefile /kaggle/working/PIPNet/pipnet/train.py
from tqdm import tqdm
import torch
import torch.nn.functional as F
import torch.optim
import torch.utils.data
import math

"""
[MODIFICATION: FUNCTION SIGNATURE]
Original: def train_pipnet(..., device, pretrain=False, ...)
Modified: def train_pipnet(..., device, args, pretrain=False, ...)
Reason: Added 'args' parameter to allow passing hyperparameter configurations 
        (specifically decay_rate, decay_start_epoch, and loss weights) into the training loop.
"""
def train_pipnet(net, train_loader, optimizer_net, optimizer_classifier, scheduler_net, scheduler_classifier, criterion, epoch, nr_epochs, device, args, pretrain=False, finetune=False, progress_prefix: str = 'Train Epoch'):

    net.train()
    
    if pretrain:
        net.module._classification.requires_grad = False
        progress_prefix = 'Pretrain Epoch'
    else:
        net.module._classification.requires_grad = True
    
    total_loss = 0.
    total_acc = 0.

    iters = len(train_loader)
    train_iter = tqdm(enumerate(train_loader), total=len(train_loader), desc=progress_prefix+'%s'%epoch, mininterval=2., ncols=0)
    
    """
    [MODIFICATION: DYNAMIC LOSS WEIGHTS]
    Original: Hardcoded weights (align=5., tanh=2., class=2.).
    Modified: Uses values from 'args' (args.w_align, args.w_tanh, args.w_class).
    Reason: To allow sensitivity analysis on the importance of different loss components 
            (Alignment vs. Uniformity vs. Classification) without modifying the source code.
    """
    if pretrain:
        # Pretraining schedule logic kept as is (ramp up for align)
        align_pf_weight = (epoch/nr_epochs)*1.
        t_weight = 5.
        cl_weight = 0.
    else:
        # Main training weights controlled by args
        align_pf_weight = args.w_align 
        t_weight = args.w_tanh
        cl_weight = args.w_class

    for i, (xs1, xs2, ys) in train_iter:        
        xs1, xs2, ys = xs1.to(device), xs2.to(device), ys.to(device)
        
        optimizer_classifier.zero_grad(set_to_none=True)
        optimizer_net.zero_grad(set_to_none=True)
        
        proto_features, pooled, out = net(torch.cat([xs1, xs2]))
        loss, acc = calculate_loss(proto_features, pooled, out, ys, align_pf_weight, t_weight, 0, cl_weight, net.module._classification.normalization_multiplier, pretrain, finetune, criterion, train_iter, print=True, EPS=1e-8)
        
        loss.backward()

        if not pretrain:
            optimizer_classifier.step()   
            scheduler_classifier.step(epoch - 1 + (i/iters))
      
        if not finetune:
            optimizer_net.step()
            scheduler_net.step() 
            
        with torch.no_grad():
            total_acc+=acc
            total_loss+=loss.item()

        """
        [MODIFICATION: DYNAMIC SPARSITY REGULARIZATION]
        Original Logic: 
            `net.module._classification.weight.copy_(torch.clamp(weight.data - 1e-3, min=0.))`
            This applied a fixed, aggressive decay at every batch step from epoch 0.
        
        Modified Logic: 
            Implemented a tunable, scheduled decay mechanism.
            1. Warm-up Check: `if epoch >= args.decay_start_epoch`
               Prevents pruning in early epochs to allow meaningful prototypes to form.
            2. Tunable Rate: Uses `args.decay_rate` instead of fixed `1e-3`.
               Allows fine-tuning the pressure based on batch size and dataset complexity.
        
        Reason: 
            The original logic caused 'Sparsity Collapse' (weights -> 0) with smaller batch sizes.
            The modified logic restores the balance between gradient updates and weight decay.
        """
        if not pretrain:
            with torch.no_grad():
                # Apply decay only after the warm-up period
                if epoch >= args.decay_start_epoch:
                    decay_rate = args.decay_rate 
                    net.module._classification.weight.copy_(torch.clamp(net.module._classification.weight.data - decay_rate, min=0.)) 
                
                # Normalization constraints (kept from original)
                net.module._classification.normalization_multiplier.copy_(torch.clamp(net.module._classification.normalization_multiplier.data, min=1.0)) 
                if net.module._classification.bias is not None:
                    net.module._classification.bias.copy_(torch.clamp(net.module._classification.bias.data, min=0.))  
                    
    train_info = dict()
    train_info['train_accuracy'] = total_acc/float(i+1)
    train_info['loss'] = total_loss/float(i+1)
    # Simplified logging for LRs to reduce overhead
    train_info['lrs_net'] = []
    train_info['lrs_class'] = []
    
    return train_info

def calculate_loss(proto_features, pooled, out, ys1, align_pf_weight, t_weight, unif_weight, cl_weight, net_normalization_multiplier, pretrain, finetune, criterion, train_iter, print=True, EPS=1e-10):
    ys = torch.cat([ys1,ys1])
    pooled1, pooled2 = pooled.chunk(2)
    pf1, pf2 = proto_features.chunk(2)

    embv2 = pf2.flatten(start_dim=2).permute(0,2,1).flatten(end_dim=1)
    embv1 = pf1.flatten(start_dim=2).permute(0,2,1).flatten(end_dim=1)
    
    a_loss_pf = (align_loss(embv1, embv2.detach())+ align_loss(embv2, embv1.detach()))/2.
    tanh_loss = -(torch.log(torch.tanh(torch.sum(pooled1,dim=0))+EPS).mean() + torch.log(torch.tanh(torch.sum(pooled2,dim=0))+EPS).mean())/2.

    if not finetune:
        loss = align_pf_weight*a_loss_pf
        loss += t_weight * tanh_loss
    
    if not pretrain:
        softmax_inputs = torch.log1p(out**net_normalization_multiplier)
        class_loss = criterion(F.log_softmax((softmax_inputs),dim=1),ys)
        
        if finetune:
            loss= cl_weight * class_loss
        else:
            loss+= cl_weight * class_loss

    acc=0.
    if not pretrain:
        ys_pred_max = torch.argmax(out, dim=1)
        correct = torch.sum(torch.eq(ys_pred_max, ys))
        acc = correct.item() / float(len(ys))
    
    if print: 
        with torch.no_grad():
            if pretrain:
                train_iter.set_postfix_str(
                f'L:{loss.item():.2f}, LA:{a_loss_pf.item():.2f}, LT:{tanh_loss.item():.2f}, nz>0.1:{torch.count_nonzero(torch.relu(pooled-0.1),dim=1).float().mean().item():.1f}', refresh=False)
            else:
                train_iter.set_postfix_str(
                f'L:{loss.item():.2f}, LC:{class_loss.item():.2f}, Ac:{acc:.2f}', refresh=False)            
    return loss, acc

def align_loss(inputs, targets, EPS=1e-12):
    assert inputs.shape == targets.shape
    loss = torch.einsum("nc,nc->n", [inputs, targets])
    loss = -torch.log(loss + EPS).mean()
    return loss

Overwriting /kaggle/working/PIPNet/pipnet/train.py


In [5]:
%%writefile /kaggle/working/PIPNet/pipnet/test.py
from tqdm import tqdm
import numpy as np
import torch
import torch.nn.functional as F
"""
[MODIFICATION: LIBRARIES]
Original: Relied on internal utility functions for metrics.
Modified: Imported sklearn metrics (accuracy, f1, recall, roc_auc, precision).
Reason: To facilitate standard metric calculation aligned with the Phase 3 protocol.
"""
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, precision_score

@torch.no_grad()
def eval_pipnet(net, test_loader, epoch, device, log=None, progress_prefix='Eval Epoch'):
    """
    [MODIFICATION: REMOVED DESTRUCTIVE WEIGHT CLAMPING]
    Original Logic: The original repository included a line inside this evaluation loop:
                    `net.module._classification.weight.copy_(torch.clamp(... - 1e-3, min=0.))`
    Modified Logic: This line has been completely REMOVED.
    Reason: 
        Weight decay/pruning should ONLY occur during the training phase. Including it in the 
        evaluation phase caused the model to 'collapse' (weights decaying to zero) rapidly 
        because evaluation runs frequently and has no gradient updates to counteract the decay.
    """
    net.eval()
    y_trues = []
    y_preds_classes = []
    y_preds_probs = [] 

    test_iter = tqdm(enumerate(test_loader), total=len(test_loader), desc=f'{progress_prefix} {epoch}', mininterval=5., ncols=0)

    for i, (xs, ys) in test_iter:
        xs, ys = xs.to(device), ys.to(device)
        
        # Forward pass
        _, pooled, out = net(xs, inference=True)
        
        # Calculate probabilities
        classifier = net.module._classification if hasattr(net, 'module') else net._classification
        probs = F.softmax(torch.log1p(out**classifier.normalization_multiplier), dim=1)
        _, preds = torch.max(out, dim=1)
        
        y_trues.extend(ys.cpu().numpy())
        y_preds_classes.extend(preds.cpu().numpy())
        
        # Store probabilities for AUC calculation
        # Handle multi-class vs binary output shapes
        if out.shape[1] > 2:
            y_preds_probs.extend(probs.cpu().numpy())
        else:
            y_preds_probs.extend(probs[:, 1].cpu().numpy()) # Probability of positive class for binary

    y_trues = np.array(y_trues)
    y_preds_classes = np.array(y_preds_classes)
    y_preds_probs = np.array(y_preds_probs)

    """
    [MODIFICATION: METRICS ALIGNMENT WITH PHASE 3 PROTOCOL]
    Original: Calculated mostly Top-1 and Top-5 accuracy manually via confusion matrix.
    Modified: Added Sensitivity (Recall), F1-Score, Precision and AUC using sklearn.
    Reason: 
        1. **Phase 3 Consistency:** To enable direct comparison with the results reported in the Phase 3 paper 
           ("An Analysis on Ensemble Learning..."), which relied on Sensitivity, F1, and AUC.
        2. **Medical Context:** Accuracy alone is insufficient for imbalanced medical datasets (COVID-19).
           'Weighted' average is used to account for class imbalance.
    """
    acc = accuracy_score(y_trues, y_preds_classes)
    f1 = f1_score(y_trues, y_preds_classes, average='weighted') 
    precision = precision_score(y_trues, y_preds_classes, average='weighted', zero_division=0)
    sensitivity = recall_score(y_trues, y_preds_classes, average='weighted') 
    
    try:
        if len(np.unique(y_trues)) == 2:
            auc = roc_auc_score(y_trues, y_preds_probs)
        else:
            auc = roc_auc_score(y_trues, y_preds_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    print(f"\n--- Metrics for Epoch {epoch} ---")
    print(f"Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {precision:.4f} | Rec/Sens: {sensitivity:.4f} | AUC: {auc:.4f}")

    # Monitor sparsity: Count non-zero prototypes to track model compactness
    classifier = net.module._classification if hasattr(net, 'module') else net._classification
    num_non_zero = torch.gt(classifier.weight, 1e-3).any(dim=0).sum().item()

    info = {
        'test_accuracy': acc, 
        'top1_accuracy': acc, 
        'top5_accuracy': f1, 
        'test_f1': f1, 
        'sensitivity': sensitivity, 
        'auc': auc,
        'almost_sim_nonzeros': 0, 
        'local_size_all_classes': 0, 
        'almost_nonzeros': 0,
        'num non-zero prototypes': num_non_zero
    }
    return info

@torch.no_grad()
def get_thresholds(net, test_loader, epoch, device, percent, log):
    """
    [NOTE] Helper function for OOD detection logic. 
    Kept consistent with the simplified logic required for Phase 5 dataloaders.
    """
    net.eval()
    all_outs = []
    for i, (xs, ys) in enumerate(test_loader):
        xs = xs.to(device)
        _, _, out = net(xs, inference=True)
        all_outs.append(out.cpu())
    
    all_outs = torch.cat(all_outs, dim=0)
    num_classes = all_outs.shape[1]
    thresholds = []
    for c in range(num_classes):
        class_outs = all_outs[:, c]
        threshold = np.percentile(class_outs.numpy(), 100 - percent)
        thresholds.append(threshold)
    
    return None, None, None, thresholds

@torch.no_grad()
def eval_ood(net, test_loader, epoch, device, thresholds):
    """
    [NOTE] Helper function to evaluate Out-of-Distribution detection performance.
    """
    net.eval()
    total_samples = 0
    accepted_samples = 0
    
    for i, (xs, ys) in enumerate(test_loader):
        xs = xs.to(device)
        _, _, out = net(xs, inference=True)
        
        for b in range(out.shape[0]):
            total_samples += 1
            max_val, max_idx = torch.max(out[b], dim=0)
            if max_val.item() > thresholds[max_idx.item()]:
                accepted_samples += 1
                
    return accepted_samples / total_samples

Overwriting /kaggle/working/PIPNet/pipnet/test.py


In [6]:
%%writefile /kaggle/working/PIPNet/util/vis_pipnet.py
import torch
import os
import numpy as np
from PIL import Image, ImageDraw as D
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm
import random

"""
[NOTE] Helper function to calculate patch size. Kept identical to original.
"""
def get_patch_size(args):
    patchsize = 32
    skip = round((args.image_size - patchsize) / (args.wshape-1))
    return patchsize, skip

"""
[NOTE] Helper function to map latent coordinates to image pixels. Kept identical to original.
"""
def get_img_coordinates(img_size, softmaxes_shape, patchsize, skip, h_idx, w_idx):
    if softmaxes_shape[1] == 26 and softmaxes_shape[2] == 26:
        h_coor_min = max(0,(h_idx-1)*skip+4)
        if h_idx < softmaxes_shape[-1]-1:
            h_coor_max = h_coor_min + patchsize
        else:
            h_coor_min -= 4
            h_coor_max = h_coor_min + patchsize
        w_coor_min = max(0,(w_idx-1)*skip+4)
        if w_idx < softmaxes_shape[-1]-1:
            w_coor_max = w_coor_min + patchsize
        else:
            w_coor_min -= 4
            w_coor_max = w_coor_min + patchsize
    else:
        h_coor_min = h_idx*skip
        h_coor_max = min(img_size, h_idx*skip+patchsize)
        w_coor_min = w_idx*skip
        w_coor_max = min(img_size, w_idx*skip+patchsize)                                    
    
    if h_idx == softmaxes_shape[1]-1:
        h_coor_max = img_size
    if w_idx == softmaxes_shape[2] -1:
        w_coor_max = img_size
    if h_coor_max == img_size:
        h_coor_min = img_size-patchsize
    if w_coor_max == img_size:
        w_coor_min = img_size-patchsize

    return h_coor_min, h_coor_max, w_coor_min, w_coor_max

"""
[MODIFICATION: DATASET PATH RETRIEVAL HELPER]
Original: Code accessed `dataset.imgs` directly, assuming it's an ImageFolder.
Modified: Added `get_image_path_from_dataset` to handle `torch.utils.data.Subset`.
Reason: 
    In Phase 5, we use Subsets for train/val/test splitting. Subsets do not expose `.imgs`.
    This function recursively finds the underlying dataset to retrieve file paths, preventing runtime errors.
"""
@torch.no_grad()
def get_image_path_from_dataset(dataset, index):
    # Handle Subset to find original file path
    if hasattr(dataset, 'indices'): # It is a Subset
        original_index = dataset.indices[index]
        # Recursively checks if the underlying dataset is also a subset or the ImageFolder
        if hasattr(dataset.dataset, 'imgs'):
             return dataset.dataset.imgs[original_index][0]
        elif hasattr(dataset.dataset, 'dataset'): # Nested Subset
             return get_image_path_from_dataset(dataset.dataset, original_index)
    elif hasattr(dataset, 'imgs'): # It is ImageFolder
        return dataset.imgs[index][0]
    return None

@torch.no_grad()                  
def visualize_topk(net, projectloader, num_classes, device, foldername, args, k=10):
    print("Visualizing prototypes for topk...", flush=True)
    dir = os.path.join(args.log_dir, foldername)
    if not os.path.exists(dir):
        os.makedirs(dir)

    patchsize, skip = get_patch_size(args)
    
    # Collect Top-K
    topks = dict()
    img_iter = tqdm(enumerate(projectloader), total=len(projectloader), desc='Collecting topk', ncols=0)
    
    # 1. Find images with highest similarity
    for i, (xs, ys) in img_iter:
        xs, ys = xs.to(device), ys.to(device)
        with torch.no_grad():
            _, pooled, _ = net(xs, inference=True)
            pooled = pooled.squeeze(0)
            
            for p in range(pooled.shape[0]):
                if pooled[p].item() > 0.1: # Only consider meaningful similarities
                    if p not in topks.keys():
                        topks[p] = []
                    if len(topks[p]) < k:
                        topks[p].append((i, pooled[p].item()))
                    else:
                        topks[p] = sorted(topks[p], key=lambda tup: tup[1], reverse=True)
                        if topks[p][-1][1] < pooled[p].item():
                            topks[p][-1] = (i, pooled[p].item())

    # 2. Save images
    tensors_per_prototype = dict()
    
    for p in topks.keys():
        tensors_per_prototype[p] = []
        for idx, score in topks[p]:
            # Retrieve image path using the helper
            dataset = projectloader.dataset
            img_path = get_image_path_from_dataset(dataset, idx)
            
            # Get patch coordinates
            xs = dataset[idx][0].unsqueeze(0).to(device)
            with torch.no_grad():
                softmaxes, _, _ = net(xs, inference=True)
            
            max_per_prototype, max_idx_per_prototype = torch.max(softmaxes, dim=0)
            max_per_prototype_h, max_idx_per_prototype_h = torch.max(max_per_prototype, dim=1)
            max_per_prototype_w, max_idx_per_prototype_w = torch.max(max_per_prototype_h, dim=1)
            
            h_idx = max_idx_per_prototype_h[p, max_idx_per_prototype_w[p]]
            w_idx = max_idx_per_prototype_w[p]
            
            try:
                image = transforms.Resize(size=(args.image_size, args.image_size))(Image.open(img_path).convert("RGB"))
                img_tensor = transforms.ToTensor()(image)
                
                h_min, h_max, w_min, w_max = get_img_coordinates(args.image_size, softmaxes.shape, patchsize, skip, h_idx, w_idx)
                img_tensor_patch = img_tensor[:, h_min:h_max, w_min:w_max]
                tensors_per_prototype[p].append(img_tensor_patch)
            except Exception as e:
                print(f"Error processing topk image: {e}")

        # Create Grid
        if len(tensors_per_prototype[p]) > 0:
            try:
                grid = torchvision.utils.make_grid(tensors_per_prototype[p], nrow=k, padding=1)
                torchvision.utils.save_image(grid, os.path.join(dir, f"grid_topk_{p}.png"))
            except Exception as e:
                pass

    return topks

"""
[MODIFICATION: SIMPLIFIED VISUALIZATION FUNCTION]
Original: Complex iteration logic with skipping.
Modified: Streamlined loop using 'get_image_path_from_dataset' and strict limit on samples.
Reason: To ensure robustness with Subset datasets and reduce execution time during experiments.
"""
@torch.no_grad()
def visualize(net, projectloader, num_classes, device, foldername, args):
    print("Visualizing prototypes prediction...", flush=True)
    dir = os.path.join(args.log_dir, foldername)
    if not os.path.exists(dir):
        os.makedirs(dir)
    
    patchsize, skip = get_patch_size(args)
    # Visualize a limited number of samples to save time
    count = 0
    limit = 50 

    for i, (xs, ys) in enumerate(projectloader):
        if count > limit: break
        count += 1
        
        xs = xs.to(device)
        with torch.no_grad():
            softmaxes, pooled, out = net(xs, inference=True)
        
        # Find max for each prototype
        img_path = get_image_path_from_dataset(projectloader.dataset, i)
        try:
            image_orig = transforms.Resize(size=(args.image_size, args.image_size))(Image.open(img_path).convert("RGB"))
        except:
            continue
        
        # For each prototype with high similarity
        for p in range(net.module._num_prototypes):
            if pooled[0, p].item() > 0.5: # Strict threshold for visualization
                max_h, max_idx_h = torch.max(softmaxes[0, p, :, :], dim=0)
                max_w, max_idx_w = torch.max(max_h, dim=0)
                
                h_idx = max_idx_h[max_idx_w].item()
                w_idx = max_idx_w.item()
                
                h_min, h_max, w_min, w_max = get_img_coordinates(args.image_size, softmaxes.shape, patchsize, skip, h_idx, w_idx)
                
                # Draw Rectangle
                img_copy = image_orig.copy()
                draw = D.Draw(img_copy)
                draw.rectangle([(w_min, h_min), (w_max, h_max)], outline='yellow', width=2)
                
                save_p_dir = os.path.join(dir, f"proto_{p}")
                os.makedirs(save_p_dir, exist_ok=True)
                img_copy.save(os.path.join(save_p_dir, f"img_{i}_sim{pooled[0,p].item():.2f}.png"))

Overwriting /kaggle/working/PIPNet/util/vis_pipnet.py


In [7]:
%%writefile /kaggle/working/PIPNet/util/visualize_prediction.py
import os
import shutil
import torch
from torchvision import transforms
from PIL import Image, ImageDraw as D
from util.vis_pipnet import get_img_coordinates, get_patch_size, get_image_path_from_dataset

"""
[MODIFICATION: INPUT TYPE CHANGE]
Original Signature: def vis_pred(net, vis_test_dir, classes, device, args)
Modified Signature: def vis_pred(net, test_loader, classes, device, args)
Reason: 
    Passing the pre-configured 'test_loader' (which respects the Subset logic) is safer 
    and more robust than passing a raw directory path and creating a new dataloader 
    internally, which could lead to data leakage or index mismatch.
"""
def vis_pred(net, test_loader, classes, device, args):
    net.eval()
    save_dir = os.path.join(args.log_dir, 'visualized_predictions')
    
    # Reset directory
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir)
    
    print("Visualizing predictions for test set...", flush=True)
    patchsize, skip = get_patch_size(args)
    
    # Limit samples per class to save time/space
    samples_per_class = 5
    class_counts = {c: 0 for c in classes}
    
    # Track global index to retrieve correct file paths from the dataset
    total_samples_seen = 0
    
    # [MODIFICATION: BATCH PROCESSING LOOP]
    # Original code crashed with "RuntimeError: Tensor with 32 elements cannot be converted to Scalar"
    # because it wasn't designed for batch_size > 1.
    # We iterate through the loader, and then iterate through items within the batch.
    for i, (xs, ys) in enumerate(test_loader):
        xs, ys = xs.to(device), ys.to(device)
        
        with torch.no_grad():
            softmaxes, pooled, out = net(xs, inference=True)
            _, preds = torch.max(out, dim=1)
        
        batch_size = xs.shape[0]
        
        # Inner loop to handle each image in the batch individually
        for b in range(batch_size):
            label_idx = ys[b].item()
            label_name = classes[label_idx]
            pred_idx = preds[b].item()
            pred_name = classes[pred_idx]
            
            # Stop if we have enough samples for this class
            if class_counts[label_name] >= samples_per_class:
                continue
            
            # [MODIFICATION: CORRECT PATH RETRIEVAL]
            # Calculate the global index in the dataset to find the original file path
            global_idx = total_samples_seen + b
            img_path = get_image_path_from_dataset(test_loader.dataset, global_idx)
            
            try:
                # Load and resize original image for visualization overlay
                image = transforms.Resize(size=(args.image_size, args.image_size))(Image.open(img_path).convert("RGB"))
            except:
                print(f"Skipping image {global_idx} due to loading error.")
                continue

            # Create specific folder for this prediction example
            img_save_folder = os.path.join(save_dir, f"{label_name}_as_{pred_name}_idx{global_idx}")
            os.makedirs(img_save_folder, exist_ok=True)
            image.save(os.path.join(img_save_folder, "original.png"))
            
            # Calculate contribution of each prototype to the prediction
            # Contribution = Weight of Proto for Predicted Class * Similarity Score of Proto
            proto_weights = net.module._classification.weight[pred_idx, :]
            contributions = proto_weights * pooled[b] # Use 'b' to access specific item in batch
            
            # Get top 3 most influential prototypes
            top_contrib, top_idx = torch.topk(contributions, 3)
            
            for k in range(3):
                pid = top_idx[k].item()
                score = top_contrib[k].item()
                
                # Ignore insignificant prototypes
                if score < 0.01: continue
                
                # Find location of this prototype in the latent map
                max_h, max_idx_h = torch.max(softmaxes[b, pid, :, :], dim=0) # Use 'b' index
                max_w, max_idx_w = torch.max(max_h, dim=0)
                h_idx = max_idx_h[max_idx_w].item()
                w_idx = max_idx_w.item()
                
                # Map latent coordinates to image pixel coordinates
                h_min, h_max, w_min, w_max = get_img_coordinates(args.image_size, softmaxes.shape, patchsize, skip, h_idx, w_idx)
                
                # Draw bounding box
                img_copy = image.copy()
                draw = D.Draw(img_copy)
                draw.rectangle([(w_min, h_min), (w_max, h_max)], outline='cyan', width=3)
                img_copy.save(os.path.join(img_save_folder, f"top{k+1}_proto{pid}_score{score:.2f}.png"))

            class_counts[label_name] += 1
        
        total_samples_seen += batch_size
        
        # Stop everything if all classes are satisfied
        if all(c >= samples_per_class for c in class_counts.values()):
            print("Collected enough samples per class. Stopping visualization.")
            break

"""
[NOTE] 'vis_pred_experiments'
Kept as a simplified placeholder since extra experimental folders are not used in the 
standard Phase 5 execution flow.
"""
def vis_pred_experiments(net, folder, classes, device, args):
    print(f"Experiments visualization skipped for folder {folder}", flush=True)

Overwriting /kaggle/working/PIPNet/util/visualize_prediction.py


In [8]:
%%writefile /kaggle/working/PIPNet/main.py
from pipnet.pipnet import PIPNet, get_network
from util.log import Log
import torch.nn as nn
from util.args import get_args, save_args, get_optimizer_nn
from util.data import get_dataloaders
from util.func import init_weights_xavier
from pipnet.train import train_pipnet
from pipnet.test import eval_pipnet
import torch
from util.vis_pipnet import visualize, visualize_topk
from util.visualize_prediction import vis_pred
import sys, os
import random
import numpy as np

def run_pipnet(args=None):
    # Set seeds for reproducibility
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)

    args = args or get_args()
    log = Log(args.log_dir)
    save_args(args, log.metadata_dir)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Prepare Data
    # [NOTE] Unpacking 8 values as returned by our custom get_dataloaders in data.py
    trainloader, trainloader_pre, _, _, projectloader, testloader, valloader, classes = get_dataloaders(args, device)
    
    # 2. Setup Network
    feature_net, add_on_layers, pool_layer, classification_layer, num_prototypes = get_network(len(classes), args)
    net = PIPNet(num_classes=len(classes), num_prototypes=num_prototypes, feature_net=feature_net, args=args, 
                 add_on_layers=add_on_layers, pool_layer=pool_layer, classification_layer=classification_layer)
    net = net.to(device)
    net = nn.DataParallel(net)    
    
    # 3. Setup Optimizer
    optimizer_net, optimizer_classifier, params_to_freeze, params_to_train, params_backbone = get_optimizer_nn(net, args)    

    # 4. Initialize Weights
    with torch.no_grad():
        net.module._add_on.apply(init_weights_xavier)
        torch.nn.init.normal_(net.module._classification.weight, mean=1.0, std=0.1) 
        torch.nn.init.constant_(net.module._multiplier, val=2.)

    criterion = nn.NLLLoss(reduction='mean').to(device)
    
    # 5. Determine Latent Shape (for patch size calculation)
    with torch.no_grad():
        xs1, xs2, ys = next(iter(trainloader))
        proto_features, _, _ = net(xs1.to(device))
        args.wshape = proto_features.shape[-1]
    
    # [MODIFICATION: LOGGING METRICS]
    # Updated headers to include Sensitivity, AUC, F1 (Phase 3 alignment)
    log.create_log('log_epoch_overview', 'epoch', 'test_acc', 'test_f1', 'sensitivity', 'auc', 'num_nonzero_prototypes', 'mean_train_acc', 'mean_train_loss')
    
    # ---------------------------------------------------------
    # PHASE 1: PRE-TRAINING (Prototypes Only)
    # ---------------------------------------------------------
    print(f"Starting Pretraining for {args.epochs_pretrain} epochs...", flush=True)
    scheduler_net = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_net, T_max=len(trainloader_pre)*args.epochs_pretrain)
    
    for epoch in range(1, args.epochs_pretrain + 1):
        for param in net.module._classification.parameters(): param.requires_grad = False
        
        # [MODIFICATION] Passing 'args' to train_pipnet for tunable sparsity
        train_pipnet(net, trainloader_pre, optimizer_net, optimizer_classifier, scheduler_net, None, criterion, epoch, args.epochs_pretrain, device, args, pretrain=True)
    
    # ---------------------------------------------------------
    # PHASE 2: MAIN TRAINING (Full Network)
    # ---------------------------------------------------------
    print(f"Starting Main Training for {args.epochs} epochs...", flush=True)
    # Re-initialize optimizers for the main phase
    optimizer_net, optimizer_classifier, _, _, _ = get_optimizer_nn(net, args)
    scheduler_net = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_net, T_max=len(trainloader)*args.epochs)
    scheduler_classifier = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_classifier, T_0=10)

    for epoch in range(1, args.epochs + 1):
        # Unfreeze layers after 'freeze_epochs'
        if epoch > args.freeze_epochs:
            for param in net.module._net.parameters():
                param.requires_grad = True
            for param in net.module._add_on.parameters():
                param.requires_grad = True
        
        # Train Step
        train_info = train_pipnet(net, trainloader, optimizer_net, optimizer_classifier, scheduler_net, scheduler_classifier, criterion, epoch, args.epochs, device, args, pretrain=False)
        
        # Eval Step
        eval_info = eval_pipnet(net, testloader, epoch, device, log)
        
        # Log Metrics
        log.log_values('log_epoch_overview', epoch, eval_info['test_accuracy'], eval_info['test_f1'], eval_info['sensitivity'], eval_info['auc'], eval_info['num non-zero prototypes'], train_info['train_accuracy'], train_info['loss'])
        
        # Save Checkpoint (every 10 epochs or at end)
        if epoch % 10 == 0 or epoch == args.epochs:
            save_path = os.path.join(args.log_dir, 'checkpoints', f'net_epoch_{epoch}.pth')
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(net.state_dict(), save_path)

    # ---------------------------------------------------------
    # VISUALIZATION PHASE
    # ---------------------------------------------------------
    """
    [MODIFICATION: VISUALIZATION CONTROL]
    Added check for 'args.visualize'.
    This prevents running expensive visualization routines during hyperparameter tuning experiments.
    """
    if args.visualize:
        print("🖼️ Visualization flag is ON. Generating prototype images...", flush=True)
        visualize_topk(net, projectloader, len(classes), device, 'visualised_prototypes_topk', args)
        visualize(net, projectloader, len(classes), device, 'visualised_prototypes', args)
        vis_pred(net, testloader, classes, device, args)
    else:
        print("Visualization skipped (use --visualize to enable).", flush=True)
    
    print("Done!", flush=True)

if __name__ == '__main__':
    args = get_args()
    os.makedirs(args.log_dir, exist_ok=True)
    # Redirect stdout/stderr to files for logging
    sys.stdout = open(os.path.join(args.log_dir, 'out.txt'), 'w')
    sys.stderr = open(os.path.join(args.log_dir, 'tqdm.txt'), 'w')
    run_pipnet(args)

Overwriting /kaggle/working/PIPNet/main.py


In [9]:
# import shutil
# import os

# # Path to the directory containing all modified files
# source_dir = '/kaggle/working/PIPNet'

# # Name and path of the output zip file
# output_filename = '/kaggle/working/pipnet_phase5_stable'

# # Create the zip file
# if os.path.exists(source_dir):
#     print("Compressing files... Please wait.")
#     shutil.make_archive(output_filename, 'zip', source_dir)
#     print(f"Success! File created: {output_filename}.zip")
#     print("Please download this file from the 'Output' section on the right sidebar.")
# else:
#     print(f"Error: Directory {source_dir} not found. Did you run the previous cells?")